In [1]:
import math
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
import torchaudio.transforms as T
from pathlib import Path
import os
import pandas as pd

# read in the dataset
FSD50K = Path('./data')
audio = FSD50K /'FSD50K.dev_audio'

# Audio Setup for TorchAudio
CLIP_TIME    = 5.94   # Seconds 
SAMPLE_RATE  = 22050  # Half of native
SAMPLE_COUNT = int(SAMPLE_RATE * CLIP_TIME)
HOP_LENGTH   = 256    # Step between windows
MEL_N        = 128    # Mel filter banks
FFT_N        = 1024   # FFT window
F_MIN        = 100    # Hz
F_MAX        = 10000  # Hz
DB_MAX       = 80.0   # DB

# Expected time frames for a 5-second clip
FRAME_COUNT = math.ceil(SAMPLE_COUNT / HOP_LENGTH)
print(f"Target shape per clip: (1, {MEL_N}, {FRAME_COUNT})")

Target shape per clip: (1, 128, 512)


In [2]:
NORM_MEAN = -22.69  # dB
NORM_STD = 21.58    # dB

mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=FFT_N,
    hop_length=HOP_LENGTH,
    n_mels=MEL_N,
    f_min=F_MIN,
    f_max=F_MAX,
    power=2.0
)
to_db = T.AmplitudeToDB(stype='power', top_db=DB_MAX)

def audio_to_log_mel(path: Path, norm_mean: float = NORM_MEAN, norm_std: float = NORM_STD) -> torch.Tensor:
    waveform, sr = torchaudio.load(path)
    waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    waveform = waveform.mean(dim=0, keepdim=True)  # Convert to mono

    waveform = waveform.repeat(1, math.ceil(SAMPLE_COUNT / waveform.shape[1]))
    midpoint = (waveform.shape[1] - SAMPLE_COUNT) // 2
    waveform = waveform[:, midpoint : midpoint + SAMPLE_COUNT]

    log_mel = to_db(mel_transform(waveform))

    # fic time axis 
    if log_mel.shape[-1] < FRAME_COUNT: log_mel = torch.nn.functional.pad(log_mel, (0, FRAME_COUNT - log_mel.shape[-1]))
    else : log_mel = log_mel[:, :, :FRAME_COUNT]

    # Normalize
    return (log_mel - norm_mean) / (norm_std + 1e-6)


In [3]:
df_eval = pd.read_csv(FSD50K / 'FSD50K.ground_truth' / 'eval.csv')
df_eval['labels']     = df_eval['labels'].apply(lambda x: x.split(','))
df_eval['audio_path'] = df_eval['fname'].apply(lambda x: os.path.join(FSD50K / 'FSD50K.eval_audio', f'{x}.wav'))

vocabulary = pd.read_csv(FSD50K / 'FSD50K.ground_truth' / 'vocabulary.csv')
vocabulary.columns = ["index", "label", "mid"]
vocabulary             = pd.read_csv(FSD50K / 'FSD50K.ground_truth' / 'vocabulary.csv')
vocabulary.columns     = ["index", "label", "mid"]
label_to_idx           = {label: idx for idx, label in enumerate(vocabulary['label'].unique())}
num_classes            = len(label_to_idx)

print(f"Num classes: {num_classes}")

Num classes: 199


In [4]:
from tqdm import tqdm
"""
# Compute normalization statistics from the evaluation set itself, then save normalized tensors.
output_dir = Path("preprocessed_eval")
output_dir.mkdir(parents=True, exist_ok=True)

# Pass 1: compute eval-set mean and std using unnormalized log-mel spectrograms.
total_sum, total_sq_sum, total_count = 0.0, 0.0, 0
for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Computing eval normalization stats"):
    path = Path(row['audio_path'])
    spec = audio_to_log_mel(path, norm_mean=0.0, norm_std=1.0)
    total_sum += spec.sum().item()
    total_sq_sum += (spec ** 2).sum().item()
    total_count += spec.numel()

NORM_MEAN = total_sum / total_count
NORM_STD = (total_sq_sum / total_count - NORM_MEAN ** 2) ** 0.5
print(f"Eval NORM_MEAN = {NORM_MEAN:.2f}, Eval NORM_STD = {NORM_STD:.2f}")

# Pass 2: save normalized eval spectrograms as .pt files.
eval_count = 0
for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Saving normalized eval spectrograms"):
    output_dir = Path("preprocessed_eval")
    output_dir.mkdir(parents=True, exist_ok=True)
    spec = audio_to_log_mel(path, norm_mean=NORM_MEAN, norm_std=NORM_STD)
    torch.save(spec, output_dir / f"{row['fname']}.pt")
    eval_count += 1

print(f"Saved {eval_count} normalized evaluation log-mel spectrograms to {output_dir}.")
"""


'\n# Compute normalization statistics from the evaluation set itself, then save normalized tensors.\noutput_dir = Path("preprocessed_eval")\noutput_dir.mkdir(parents=True, exist_ok=True)\n\n# Pass 1: compute eval-set mean and std using unnormalized log-mel spectrograms.\ntotal_sum, total_sq_sum, total_count = 0.0, 0.0, 0\nfor _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Computing eval normalization stats"):\n    path = Path(row[\'audio_path\'])\n    spec = audio_to_log_mel(path, norm_mean=0.0, norm_std=1.0)\n    total_sum += spec.sum().item()\n    total_sq_sum += (spec ** 2).sum().item()\n    total_count += spec.numel()\n\nNORM_MEAN = total_sum / total_count\nNORM_STD = (total_sq_sum / total_count - NORM_MEAN ** 2) ** 0.5\nprint(f"Eval NORM_MEAN = {NORM_MEAN:.2f}, Eval NORM_STD = {NORM_STD:.2f}")\n\n# Pass 2: save normalized eval spectrograms as .pt files.\neval_count = 0\nfor _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Saving normalized eval spectrogra

In [5]:
#evaluate metrics function
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                              average_precision_score, roc_auc_score,
                              precision_recall_curve, auc)
from scipy.stats import norm

def evaluate_metrics(model, loader, threshold=0.3):
    model.eval()
    all_preds, all_labels_list, all_probs_list = [], [], []
    with torch.no_grad():
        for specs, labels in loader:
            probs = torch.sigmoid(model(specs.to(device))).cpu()
            all_preds.append((probs > threshold).float())
            all_labels_list.append(labels)
            all_probs_list.append(probs)
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels_list).numpy()
    all_probs  = torch.cat(all_probs_list).numpy()
    return all_preds, all_labels, all_probs

def tune_thresholds(all_labels, all_probs):
    best_thresholds = []
    for i in range(all_labels.shape[1]):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (all_probs[:, i] > t).astype(float)
            f1 = f1_score(all_labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(best_t)
    return np.array(best_thresholds)

def compute_all_metrics(all_labels, all_probs):
    # Tuned F1
    thresholds     = tune_thresholds(all_labels, all_probs)
    tuned_preds    = (all_probs > thresholds).astype(float)
    f1_micro       = f1_score(all_labels, tuned_preds, average='micro', zero_division=0)
    f1_macro       = f1_score(all_labels, tuned_preds, average='macro', zero_division=0)

    # mAP
    map_score      = average_precision_score(all_labels, all_probs, average='macro')

    # d'
    valid          = (all_labels.sum(axis=0) > 0) & (all_labels.sum(axis=0) < len(all_labels))
    roc_auc        = roc_auc_score(all_labels[:, valid], all_probs[:, valid], average='macro')
    dp             = np.sqrt(2) * norm.ppf(roc_auc)

    # lωlrap
    has_pos        = all_labels.sum(axis=1) > 0
    labels_pos     = all_labels[has_pos]
    probs_pos      = all_probs[has_pos]
    per_sample_ap, per_sample_w = [], []
    for i in range(len(labels_pos)):
        pos_cls    = np.where(labels_pos[i] > 0)[0]
        ranked     = np.argsort(-probs_pos[i])
        precisions, num_correct = [], 0
        for rank, cls in enumerate(ranked):
            if cls in pos_cls:
                num_correct += 1
                precisions.append(num_correct / (rank + 1))
            if num_correct == len(pos_cls):
                break
        per_sample_ap.append(np.mean(precisions))
        per_sample_w.append(len(pos_cls))
    lwlrap_score   = np.sum(np.array(per_sample_w) * np.array(per_sample_ap)) / np.sum(per_sample_w)

    # mAP 6 families
    families       = ['Music', 'Human_voice', 'Domestic_sounds_and_home_sounds',
                      'Animal', 'Vehicle', 'Natural_sounds']
    family_idx     = [label_to_idx[f] for f in families if f in label_to_idx]
    map_families   = average_precision_score(all_labels[:, family_idx],
                                             all_probs[:, family_idx], average='macro')

    print(f"F1 Micro (tuned): {f1_micro:.4f}  |  F1 Macro (tuned): {f1_macro:.4f}")
    print(f"mAP:    {map_score:.3f}  (paper CRNN: 0.417, VGG-like: 0.434)")
    print(f"d':     {dp:.3f}         (paper CRNN: 2.068, VGG-like: 2.167)")
    print(f"lωlrap: {lwlrap_score:.3f}  (paper CRNN: 0.519, VGG-like: 0.514)")
    print(f"mAP (6 families): {map_families:.4f}")

    return all_preds, all_labels, all_probs

In [6]:
from pathlib import Path

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class FSD50KDataset(Dataset):
    def __init__(self, df, label_to_idx, num_classes, split='eval'):
        self.df           = df.reset_index(drop=True)
        self.num_classes  = num_classes
        self.split        = split

        self.labels = torch.zeros(len(self.df), num_classes)
        for i, row in self.df.iterrows():
            for lbl in row['labels']:
                lbl = lbl.strip()
                if lbl in label_to_idx:
                    self.labels[i, label_to_idx[lbl]] = 1.0
        self.fnames = self.df['fname'].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        audio_path = Path(self.df.iloc[idx]['audio_path'])
        spec = audio_to_log_mel(audio_path, NORM_MEAN, NORM_STD)
        return spec, self.labels[idx]

class AudioCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# load my best model and evaluate final metrics on the evaluation set
path = 'best_oversampled_model.pt'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AudioCNN(num_classes=num_classes).to(device)
model.load_state_dict(torch.load(path, map_location=device))
# predict on evaluation set and compute metrics
eval_dataset = FSD50KDataset(df_eval, label_to_idx, num_classes, split='eval')
eval_loader  = DataLoader(eval_dataset, batch_size=32, shuffle=False, num_workers=0)
all_preds, all_labels, all_probs = evaluate_metrics(model, eval_loader)
print(f"Final evaluation on best model from {path}:")
_ = compute_all_metrics(all_labels, all_probs)

Final evaluation on best model from best_oversampled_model.pt:
F1 Micro (tuned): 0.5260  |  F1 Macro (tuned): 0.4458
mAP:    0.414  (paper CRNN: 0.417, VGG-like: 0.434)
d':     2.097         (paper CRNN: 2.068, VGG-like: 2.167)
lωlrap: 0.610  (paper CRNN: 0.519, VGG-like: 0.514)
mAP (6 families): 0.7068


In [7]:
#evaluate the unhelped model:
unhelped = 'best_unhelped_model.pt'
model_unhelped = AudioCNN(num_classes=num_classes).to(device)
model_unhelped.load_state_dict(torch.load(unhelped, map_location=device))
all_preds_unhelped, all_labels_unhelped, all_probs_unhelped = evaluate_metrics(model_unhelped, eval_loader)
print(f"Evaluation of unhelped model from {unhelped}:")
_ = compute_all_metrics(all_labels_unhelped, all_probs_unhelped)

Evaluation of unhelped model from best_unhelped_model.pt:
F1 Micro (tuned): 0.4699  |  F1 Macro (tuned): 0.3502
mAP:    0.327  (paper CRNN: 0.417, VGG-like: 0.434)
d':     1.934         (paper CRNN: 2.068, VGG-like: 2.167)
lωlrap: 0.555  (paper CRNN: 0.519, VGG-like: 0.514)
mAP (6 families): 0.6455


In [ ]:
#best weighted model 
weighted = 'best_weighted_model.pt'
model_weighted = AudioCNN(num_classes=num_classes).to(device)
model_weighted.load_state_dict(torch.load(weighted, map_location=device))
all_preds_weighted, all_labels_weighted, all_probs_weighted = evaluate_metrics(model_weighted, eval_loader)
print(f"Evaluation of weighted model from {weighted}:")
_ = compute_all_metrics(all_labels_weighted, all_probs_weighted)